In [1]:
import requests
from dotenv import load_dotenv
import os
import time
import pandas as pd

load_dotenv()
ELSEVIER_API_KEY = os.getenv("ELSEVIER_API")

BASE_URL_CITED_BY_SCOPUS = "https://api.elsevier.com/content/search/scopus"
INPUT_CSV_FILE="fla_csvs/continent.csv"
OUTPUT_CSV_FILE="fla_csvs/cites.csv"
REQUEST_DELAY_SECONDS=0.25


In [3]:
def get_total_citation_count_citedby_scopus(doi: str, api_key: str) -> int | None:
    """
    Fetches the total citation count for a given DOI using the Elsevier Scopus
    "Cited By in Scopus" functionality via the Scopus Search API.

    Args:
        doi (str): The Digital Object Identifier (DOI) of the document.
        api_key (str): Your Elsevier API key.

    Returns:
        int | None: The total citation count as an integer, or None if an error occurs.
    """
    if not api_key:
        print("Error: ELSEVIER_API_KEY not found. Please set it in your .env file.")
        return None

    # Construct the query string to find documents that cite this DOI
    query_string = f"DOI({doi})"

    # Parameters for the API request
    params = {
        "query": query_string,
        "apiKey": api_key,
        "field":"citedby-count",
        "count": 0 # Request 0 results, just the total count (more efficient)by
    }

    try:
        # Make the GET request to the API
        response = requests.get(BASE_URL_CITED_BY_SCOPUS, params=params)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

        # Parse the JSON response
        data = response.json()

        # The total number of citing documents is found in 'opensearch:totalResults'
        if 'search-results' in data and 'opensearch:totalResults' in data['search-results']:
            total_results_str = data['search-results']['opensearch:totalResults']
            try:
                return int(data['search-results']['entry'][0]['citedby-count'])
            except ValueError:
                print(f"Warning: Could not convert total results '{total_results_str}' to integer for DOI: {doi}")
                return None
        else:
            print(f"Error: Unexpected response structure for DOI {doi}. Could not find total results. Full response:")
            print(data)
            return None

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred for DOI {doi}: {http_err}")
        if response.status_code == 401:
            print("  Check your API key. It might be invalid or expired.")
        elif response.status_code == 403:
            print("  Forbidden: Your API key might not have permission for this resource or you've exceeded your quota.")
            print("  The 'Cited By in Scopus' feature uses the Scopus Search API, which may have specific access rules.")
        elif response.status_code == 404:
            print(f"  DOI not found or no citation data available for {doi}.")
        elif response.status_code == 429: # Specifically handle Too Many Requests
            print("\n  !!! RATE LIMIT EXCEEDED !!!")
            print("  Response Headers:")
            for header, value in response.headers.items():
                print(f"    {header}: {value}")
            if 'X-RateLimit-Reset' in response.headers:
                reset_time_str = response.headers['X-RateLimit-Reset']
                try:
                    # Attempt to parse as Unix timestamp (common for rate limits)
                    reset_timestamp = int(reset_time_str)
                    print(f"  You can resubmit requests after: {time.ctime(reset_timestamp)}")
                except ValueError:
                    # If not a Unix timestamp, print as is
                    print(f"  Rate limit will reset at: {reset_time_str}")
                print(f"  Consider increasing the REQUEST_DELAY_SECONDS to avoid this.")
            else:
                print("  X-RateLimit-Reset header not found. Please consult API documentation.")
        return None
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Connection error occurred for DOI {doi}: {conn_err}")
        print("  Please check your internet connection and ensure the URL is correctly formed.")
        return None
    except requests.exceptions.Timeout as timeout_err:
        print(f"Timeout error occurred for DOI {doi}: {timeout_err}")
        return None
    except requests.exceptions.RequestException as req_err:
        print(f"An unexpected request error occurred for DOI {doi}: {req_err}")
        print("  This might be due to a malformed URL or other request-related issue.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred for DOI {doi}: {e}")
        return None

In [4]:


df = pd.read_csv(INPUT_CSV_FILE)
print(f"Successfully loaded '{INPUT_CSV_FILE}'. Shape: {df.shape}")

citation_counts = []
# Iterate through each DOI in the specified column
for i, doi in enumerate(df["doi"]):
    # Convert DOI to string to handle potential non-string types (e.g., NaN)
    doi_str = str(doi).strip() if pd.notna(doi) else ""
    if not doi_str:
        print(f"\nSkipping row {i+1}: Empty or invalid DOI found.")
        citation_counts.append(None)
        continue
    count = get_total_citation_count_citedby_scopus(doi_str, ELSEVIER_API_KEY)
    citation_counts.append(count)
    if count is not None:
        print(f"Total citations for {doi_str}: {count}")
    else:
        print(f"Could not retrieve total citation count for {doi_str}.")
    # Introduce a delay between requests to avoid rate limits
    if i < len(df) - 1: # Don't delay after the last request
        print(f"  Waiting for {REQUEST_DELAY_SECONDS} seconds before next request...")
        time.sleep(REQUEST_DELAY_SECONDS)

# Add the citation counts as a new column to the DataFrame
df['citation_count'] = citation_counts
# Save the updated DataFrame to a new CSV file
df.to_csv(OUTPUT_CSV_FILE, index=False)
print(f"\n--- Processing Complete ---")
print(f"Updated data saved to '{OUTPUT_CSV_FILE}'.")
print(f"First 5 rows of the updated DataFrame:\n{df.head()}")


Successfully loaded 'fla_csvs/continent.csv'. Shape: (5, 41)
Total citations for 10.1016/j.tra.2017.11.003: 61
  Waiting for 0.25 seconds before next request...
Total citations for 10.1016/j.tra.2018.02.017: 109
  Waiting for 0.25 seconds before next request...
Total citations for 10.1016/j.tra.2018.04.028: 27
  Waiting for 0.25 seconds before next request...
Total citations for 10.1016/j.tra.2018.05.004: 20
  Waiting for 0.25 seconds before next request...
Total citations for 10.1016/j.tra.2018.05.018: 93

--- Processing Complete ---
Updated data saved to 'fla_csvs/cites.csv'.
First 5 rows of the updated DataFrame:
   Unnamed: 0.3  Unnamed: 0.2  Unnamed: 0.1  Unnamed: 0  year  month  \
0             0             0             0           0  2019     10   
1             1             1             1           1  2019      7   
2             2             2             2           2  2020      7   
3             3             3             3           3  2019      6   
4             4 

6